<a href="https://colab.research.google.com/github/comfyhavana/SCT_ML_3/blob/main/SCT_ML_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
!unzip data.zip -d /content/sample_data/

Archive:  data.zip
replace /content/sample_data/data/cat/00000-4122619873.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: /content/sample_data/data/cat/00000-4122619873.png  
  inflating: /content/sample_data/data/cat/00001-4122619874.png  
  inflating: /content/sample_data/data/cat/00002-4122619875.png  
  inflating: /content/sample_data/data/cat/00003-4122619876.png  
  inflating: /content/sample_data/data/cat/00004-4122619877.png  
  inflating: /content/sample_data/data/cat/00005-4122619878.png  
  inflating: /content/sample_data/data/cat/00006-4122619879.png  
  inflating: /content/sample_data/data/cat/00007-4122619880.png  
  inflating: /content/sample_data/data/cat/00008-4122619881.png  
  inflating: /content/sample_data/data/cat/00009-4122619882.png  
  inflating: /content/sample_data/data/cat/00010-4122619883.png  
  inflating: /content/sample_data/data/cat/00011-4122619884.png  
  inflating: /content/sample_data/data/cat/00012-4122619885.png  
  inflating: /content/s

In [22]:
!pip install scikit-image

In [26]:
import os
import cv2
import numpy as np
from skimage.feature import hog  # <--- ADD THIS LINE
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

In [27]:
# Step 1: Set paths based on your unzipped location
DATA_DIR = "/content/sample_data/data"
CATEGORIES = ["cat", "dog"]

data = []
labels = []

print("Loading and preprocessing images...")

for category in CATEGORIES:
    path = os.path.join(DATA_DIR, category)
    label = CATEGORIES.index(category)  # 0 for cat, 1 for dog

    if not os.path.exists(path):
        print(f"Directory not found: {path}")
        continue

    for img_name in os.listdir(path):
        try:
            img_path = os.path.join(path, img_name)
            # Load images in color (RGB) and resize to (128, 128, 3)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE) # Load in grayscale
            if img is None:
                continue
            img = cv2.resize(img, (64, 64))  # Uniform sizing for MobileNetV2

            # Extract HOG features
            # orientations=9, pixels_per_cell=(8,8), cells_per_block=(2,2) are standard robust parameters
            features = hog(
                img,
                orientations=9,
                pixels_per_cell=(8, 8),
                cells_per_block=(2, 2),
                visualize=False
            )

            data.append(features)
            labels.append(label)
        except Exception as e:
            # Skip any corrupted images if present
            pass

Loading and preprocessing images...


In [28]:
# Step 2: Convert to numpy arrays
X = np.array(data) # X is now (N, 128, 128, 3)
y = np.array(labels)

print(f"HOG Feature Extraction Complete! Shape of X: {X.shape}")
# Each 64x64 image is now condensed into an optimized 1,764-dimensional feature vector

print(f"Dataset loaded. Total images: {len(X)}")
print(f"Shape of feature matrix X: {X.shape}")  # Should be (Total Images, 4096)

HOG Feature Extraction Complete! Shape of X: (1000, 1764)
Dataset loaded. Total images: 1000
Shape of feature matrix X: (1000, 1764)


In [29]:
# -------------------------------------------------------------------------
# Step 3: Split Dataset into Training & Testing Sets (80/20)
# -------------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

Train set size: 800 samples
Test set size: 200 samples


In [30]:
# -------------------------------------------------------------------------
# Step 4: Train and Evaluate the Raw SVM Classifier (No Tuning)
# -------------------------------------------------------------------------
print("\nTraining Support Vector Machine (SVM) on raw pixels...")
# Utilizing the RBF (Radial Basis Function) kernel with default settings (C=1.0)
# which is generally much better than a linear kernel when dealing with raw, non-linear pixel data
svm_model = SVC(kernel='rbf', C=5.0, gamma='scale', random_state=42)
svm_model.fit(X_train, y_train)

# Make Predictions
y_pred = svm_model.predict(X_test)

# Evaluate Results
print("\n--- Evaluation Summary ---")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=CATEGORIES))


Training Support Vector Machine (SVM) on raw pixels...

--- Evaluation Summary ---
Accuracy Score: 86.50%

Classification Report:
              precision    recall  f1-score   support

         cat       0.88      0.84      0.86       100
         dog       0.85      0.89      0.87       100

    accuracy                           0.86       200
   macro avg       0.87      0.86      0.86       200
weighted avg       0.87      0.86      0.86       200

